# H3 Police Response Coverage

This notebook computes a complete H3 response-coverage grid for Western Australia.

Spatial joins, attribute lookups, and aggregations run as SQL in [DuckDB](https://duckdb.org/) with the `spatial` extension. Graph traversal (reachability and Dijkstra) stays in Python. Every intermediate result is a table in `data/processed/Police_Response_Coverage.duckdb`, so a run can be explored afterwards with plain SQL.

The workflow:

1. Load and validate roads, stations, and LGA boundaries with `ST_Read`, projecting to GDA94 / MGA zone 50 (EPSG:28350) with `ST_Transform`. Road speeds come from a `road_speeds` lookup table joined on `NETWORK_TYPE`.
2. Dense-polyfill every H3 cell that intersects the WA LGA boundary union (`ST_Union_Agg`) at **resolution 7** (`h3.polygon_to_cells`, not a sparse lat/lon lattice).
3. Build a weighted, bidirectional road graph using segment length and configurable travel speeds.
4. Snap each cell centroid to the nearest **station-reachable** road segment within `MAX_SAMPLE_DISTANCE_METRES` (5 km) with an `ST_DWithin` spatial join; if none, no snap → unreachable (orphaned Main Roads components are ignored). Exact distance ties go to the lowest `segment_id`.
5. Calculate shortest-road response times from every police station to each cell's assigned road segment(s), aggregating per cell in SQL.
6. Classify each cell by its fastest response time: green/yellow/orange/red for reachable times, **`unreachable`** when no station can reach the cell (including no station-reachable road within the snap cap). Red means reachable but >15 min only.

**Perth land-strict gate membership:** cells with no station-reachable road within 5 km are outside the strict land gate (same threshold as the snap cap)—see `docs/response-coverage-improvements.md`.
7. Export the complete coverage as GeoJSON (closed rings) and PMTiles (`-Z0 -z12`, no tiny-polygon drop).

> **Important:** The source road data does not contain a travel-speed field. The speed profile below is configurable and should be reviewed before operational use.

## Environment setup

Run the following command from the repository root before opening this notebook:

```bash
uv sync
uv run python -m jupyter notebook
```

The project already declares `duckdb`, `h3`, `shapely`, and `pyarrow` through `pyproject.toml`. DuckDB downloads its `spatial` extension on the first run, so that run needs network access. The `tippecanoe` executable must be installed separately and should be available on the shell path.

In [ ]:
from __future__ import annotations

import heapq
import json
import shutil
import subprocess
from collections import defaultdict, deque
from pathlib import Path
from typing import Any

import duckdb
import h3
import pyarrow as pa
import shapely
from h3 import LatLngPoly
from shapely.geometry import Polygon


REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "data" / "raw").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

RAW_DIR = REPOSITORY_ROOT / "data" / "raw"
PROCESSED_DIR = REPOSITORY_ROOT / "data" / "processed"
ROAD_PATH = RAW_DIR / "Road_Network.geojson"
STATION_PATH = RAW_DIR / "WAPoliceForceFacilities_SHP" / "Police_Facilities.shp"
BOUNDARY_PATH = RAW_DIR / "LGA_Boundaries_LGATE_233_WA_GDA2020_Public.geojson"
DATABASE_PATH = PROCESSED_DIR / "Police_Response_Coverage.duckdb"

H3_RESOLUTION = 7
MAX_SAMPLE_DISTANCE_METRES = 5000.0
MAX_RESPONSE_MINUTES = 15.0

# Each input uses a different CRS, so ST_Transform is given the source CRS
# explicitly. Distances and lengths are measured in PROJECTED_CRS metres.
ROAD_CRS = "EPSG:4326"
STATION_CRS = "EPSG:4283"
BOUNDARY_CRS = "EPSG:7844"
PROJECTED_CRS = "EPSG:28350"

# Review these values before using the output operationally. The default profile
# is an illustrative estimate, not a measured policing response-time model.
ROAD_SPEED_KMH: dict[str, float] = {
    "State Road": 80.0,
    "Main Roads Controlled Path": 80.0,
    "Local Road": 50.0,
    "Miscellaneous Road": 40.0,
    "Proposed Road": 30.0,
    "Crossover": 60.0,
    "Unknown": 40.0,
}

RESPONSE_CLASSIFICATIONS = [
    {"label": "green", "min": 0.0, "max": 5.0, "color": "#2E7D32"},
    {"label": "yellow", "min": 5.0, "max": 10.0, "color": "#F9A825"},
    {"label": "orange", "min": 10.0, "max": 15.0, "color": "#EF6C00"},
    {"label": "red", "min": 15.0, "max": float("inf"), "color": "#C62828"},
]


def connect() -> duckdb.DuckDBPyConnection:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect(DATABASE_PATH)
    con.install_extension("spatial")
    con.load_extension("spatial")
    con.execute("SET VARIABLE max_snap_metres = ?", [MAX_SAMPLE_DISTANCE_METRES])
    return con


def load_inputs(con: duckdb.DuckDBPyConnection) -> None:
    """Load, validate, and project roads, stations, and boundaries into DuckDB tables."""
    for path in (ROAD_PATH, STATION_PATH, BOUNDARY_PATH):
        if not path.exists():
            raise FileNotFoundError(f"Required input layer not found: {path}")

    con.execute(
        "CREATE OR REPLACE TABLE road_speeds (network_type VARCHAR PRIMARY KEY, speed_kmh DOUBLE)"
    )
    con.executemany("INSERT INTO road_speeds VALUES (?, ?)", list(ROAD_SPEED_KMH.items()))

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE raw_roads AS
        SELECT
            r.OGC_FID AS road_fid,
            r.START_NODE_NO AS start_node_no,
            r.END_NODE_NO AS end_node_no,
            coalesce(r.NETWORK_TYPE, 'Unknown') AS network_type,
            coalesce(speeds.speed_kmh, {ROAD_SPEED_KMH["Unknown"]}) AS speed_kmh,
            r.geom AS geom,
            ST_Transform(r.geom, '{ROAD_CRS}', '{PROJECTED_CRS}', always_xy := true) AS projected_geom
        FROM ST_Read('{ROAD_PATH}') AS r
        LEFT JOIN road_speeds AS speeds ON speeds.network_type = coalesce(r.NETWORK_TYPE, 'Unknown')
    """)

    # segment_id is the 0-based position among valid roads, in source-file order.
    con.execute("""
        CREATE OR REPLACE TABLE roads AS
        SELECT
            row_number() OVER (ORDER BY road_fid) - 1 AS segment_id,
            *,
            ST_Length(projected_geom) / 1000.0 / speed_kmh * 60.0 AS weight_minutes
        FROM raw_roads
        WHERE start_node_no IS NOT NULL
          AND end_node_no IS NOT NULL
          AND ST_IsValid(geom)
          AND ST_Length(projected_geom) > 0
          AND speed_kmh > 0
    """)

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE raw_stations AS
        SELECT
            OGC_FID AS station_id,
            NAME AS name,
            geom,
            ST_Transform(geom, '{STATION_CRS}', '{PROJECTED_CRS}', always_xy := true) AS projected_geom
        FROM ST_Read('{STATION_PATH}')
    """)
    con.execute("""
        CREATE OR REPLACE TABLE stations AS
        SELECT * FROM raw_stations
        WHERE ST_IsValid(geom) AND ST_GeometryType(geom) = 'POINT'
    """)

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE raw_boundaries AS
        SELECT
            name,
            geom,
            ST_Transform(geom, '{BOUNDARY_CRS}', '{PROJECTED_CRS}', always_xy := true) AS projected_geom
        FROM ST_Read('{BOUNDARY_PATH}')
    """)
    con.execute("""
        CREATE OR REPLACE TABLE boundaries AS
        SELECT * FROM raw_boundaries
        WHERE ST_IsValid(geom) AND ST_Area(projected_geom) > 0
    """)

    counts = dict(con.execute("""
        SELECT 'Road rows', count(*) FROM raw_roads
        UNION ALL SELECT 'Valid road rows', count(*) FROM roads
        UNION ALL SELECT 'Station rows', count(*) FROM raw_stations
        UNION ALL SELECT 'Valid station rows', count(*) FROM stations
        UNION ALL SELECT 'Boundary rows', count(*) FROM raw_boundaries
        UNION ALL SELECT 'Valid boundary rows', count(*) FROM boundaries
    """).fetchall())
    print(f"Road rows: {counts['Road rows']}")
    print(f"Valid road rows: {counts['Valid road rows']}")
    print(f"Excluded road rows: {counts['Road rows'] - counts['Valid road rows']}")
    print(f"Station rows: {counts['Station rows']}")
    print(f"Valid station rows: {counts['Valid station rows']}")
    print(f"Boundary rows: {counts['Boundary rows']}")
    print(f"Valid boundary rows: {counts['Valid boundary rows']}")

    if not (counts["Valid road rows"] and counts["Valid station rows"] and counts["Valid boundary rows"]):
        raise ValueError("At least one valid road, station, and boundary feature are required.")

    for label, table in (("road", "roads"), ("station", "stations")):
        extent = con.execute(f"""
            SELECT ST_XMin(e), ST_YMin(e), ST_XMax(e), ST_YMax(e)
            FROM (SELECT ST_Extent_Agg(projected_geom) AS e FROM {table})
        """).fetchone()
        print(f"Projected {label} extent: {list(extent)}")

In [ ]:
def build_road_graph(con: duckdb.DuckDBPyConnection) -> dict[int, list[tuple[int, float]]]:
    """Number road nodes in SQL, then build a bidirectional weighted graph in Python."""
    con.execute("""
        CREATE OR REPLACE TABLE road_nodes AS
        SELECT node_no, row_number() OVER (ORDER BY node_no) - 1 AS node_id
        FROM (
            SELECT start_node_no AS node_no FROM roads
            UNION
            SELECT end_node_no FROM roads
        )
    """)
    con.execute("""
        CREATE OR REPLACE TABLE segments AS
        SELECT r.segment_id, s.node_id AS start_id, e.node_id AS end_id,
               r.network_type, r.speed_kmh, r.weight_minutes, r.projected_geom
        FROM roads AS r
        JOIN road_nodes AS s ON s.node_no = r.start_node_no
        JOIN road_nodes AS e ON e.node_no = r.end_node_no
    """)

    graph: dict[int, list[tuple[int, float]]] = defaultdict(list)
    edges = con.execute(
        "SELECT start_id, end_id, weight_minutes FROM segments ORDER BY segment_id"
    ).fetchall()
    for start_id, end_id, weight_minutes in edges:
        graph[start_id].append((end_id, weight_minutes))
        graph[end_id].append((start_id, weight_minutes))

    for node_id in graph:
        graph[node_id] = sorted(set(graph[node_id]))

    print(f"Graph nodes: {con.execute('SELECT count(*) FROM road_nodes').fetchone()[0]}")
    print(f"Road segments: {len(edges)}")
    print(f"Graph edges: {sum(len(neighbors) for neighbors in graph.values()) // 2}")
    return dict(graph)


def create_station_sources(con: duckdb.DuckDBPyConnection) -> list[tuple[int, int, float, float]]:
    """Connect each station to the nearest road segment and its endpoints."""
    con.execute("""
        CREATE OR REPLACE TABLE station_sources AS
        WITH nearest_segment AS (
            SELECT st.station_id, st.projected_geom AS station_geom,
                   seg.start_id, seg.end_id, seg.speed_kmh, seg.projected_geom AS segment_geom
            FROM stations AS st
            JOIN segments AS seg
              ON ST_DWithin(st.projected_geom, seg.projected_geom, getvariable('max_snap_metres'))
            QUALIFY row_number() OVER (
                PARTITION BY st.station_id
                ORDER BY ST_Distance(st.projected_geom, seg.projected_geom), seg.segment_id
            ) = 1
        ),
        endpoints AS (
            SELECT station_id, start_id AS node_id, speed_kmh,
                   ST_Distance(station_geom, ST_StartPoint(segment_geom)) / 1000.0 AS distance_km
            FROM nearest_segment
            UNION ALL
            SELECT station_id, end_id, speed_kmh,
                   ST_Distance(station_geom, ST_EndPoint(segment_geom)) / 1000.0
            FROM nearest_segment
        )
        SELECT node_id, station_id,
               greatest(0.01, distance_km / speed_kmh * 60.0) AS access_minutes,
               distance_km
        FROM endpoints
        WHERE distance_km > 0 AND distance_km <= getvariable('max_snap_metres') / 1000.0
    """)
    return con.execute(
        "SELECT node_id, station_id, access_minutes, distance_km FROM station_sources "
        "ORDER BY station_id, node_id"
    ).fetchall()


def ring_latlng(poly: Polygon) -> list[tuple[float, float]]:
    """Exterior ring as (lat, lng) for h3 LatLngPoly (drop closing vertex)."""
    coords = list(poly.exterior.coords)
    if coords[0] == coords[-1]:
        coords = coords[:-1]
    return [(lat, lng) for lng, lat in coords]


def polygon_to_cells_safe(poly: Polygon, res: int) -> set[str]:
    """Dense polyfill a Shapely polygon (holes as *variadic* LatLngPoly args)."""
    exterior = ring_latlng(poly)
    if len(exterior) < 3:
        return set()
    holes: list[list[tuple[float, float]]] = []
    for hole in poly.interiors:
        coords = list(hole.coords)
        if coords[0] == coords[-1]:
            coords = coords[:-1]
        hole_ring = [(lat, lng) for lng, lat in coords]
        if len(hole_ring) >= 3:
            holes.append(hole_ring)
    return set(h3.polygon_to_cells(LatLngPoly(exterior, *holes), res))


def generate_full_grid(con: duckdb.DuckDBPyConnection) -> list[str]:
    """Dense H3 polyfill of the WA boundary union, stored in the h3_cells table."""
    union_wkb = con.execute("SELECT ST_AsWKB(ST_Union_Agg(geom)) FROM boundaries").fetchone()[0]
    boundary_geometry = shapely.from_wkb(bytes(union_wkb))

    if boundary_geometry.geom_type == "Polygon":
        polys = [boundary_geometry]
    elif boundary_geometry.geom_type == "MultiPolygon":
        polys = list(boundary_geometry.geoms)
    else:
        polys = [
            geom
            for geom in getattr(boundary_geometry, "geoms", [])
            if geom.geom_type == "Polygon"
        ]

    cells: set[str] = set()
    for poly in polys:
        cells.update(polygon_to_cells_safe(poly, H3_RESOLUTION))
    full_grid = sorted(cells)

    centroids = [h3.cell_to_latlng(cell) for cell in full_grid]
    grid = pa.table({
        "cell_id": list(range(len(full_grid))),
        "h3_cell": full_grid,
        "latitude": [latitude for latitude, _ in centroids],
        "longitude": [longitude for _, longitude in centroids],
    })
    con.execute(f"""
        CREATE OR REPLACE TABLE h3_cells AS
        SELECT cell_id, h3_cell,
               ST_Transform(ST_Point(longitude, latitude), 'EPSG:4326', '{PROJECTED_CRS}',
                            always_xy := true) AS projected_centroid
        FROM grid
    """)
    return full_grid


def nodes_reachable_from_stations(
    graph: dict[int, list[tuple[int, float]]],
    station_sources: list[tuple[int, int, float, float]],
) -> set[int]:
    """Undirected BFS from every station→road source node."""
    reachable: set[int] = set()
    for source_node in {source[0] for source in station_sources}:
        if source_node in reachable:
            continue
        queue: deque[int] = deque([source_node])
        seen = {source_node}
        while queue:
            node = queue.popleft()
            for neighbor, _weight in graph.get(node, []):
                if neighbor not in seen:
                    seen.add(neighbor)
                    queue.append(neighbor)
        reachable |= seen
    return reachable


def assign_h3_cells(con: duckdb.DuckDBPyConnection, station_reachable_nodes: set[int]) -> None:
    """Snap each cell to the nearest station-reachable road segment.

    Only segments whose endpoints are reachable from a police station are
    considered. Centroids with no such segment within
    ``MAX_SAMPLE_DISTANCE_METRES`` get no cell_segments row (unreachable) —
    orphaned Main Roads components are not used for false network times.
    Exact distance ties go to the lowest segment_id so runs are reproducible.
    """
    reachable = pa.table({"node_id": sorted(station_reachable_nodes)})
    con.execute("""
        CREATE OR REPLACE TABLE reachable_segments AS
        SELECT * FROM segments
        WHERE start_id IN (SELECT node_id FROM reachable)
           OR end_id IN (SELECT node_id FROM reachable)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cell_segments AS
        SELECT c.cell_id, seg.segment_id, seg.start_id, seg.end_id
        FROM h3_cells AS c
        JOIN reachable_segments AS seg
          ON ST_DWithin(c.projected_centroid, seg.projected_geom, getvariable('max_snap_metres'))
        QUALIFY row_number() OVER (
            PARTITION BY c.cell_id
            ORDER BY ST_Distance(c.projected_centroid, seg.projected_geom), seg.segment_id
        ) = 1
    """)

In [ ]:
def dijkstra(
    graph: dict[int, list[tuple[int, float]]],
    source: int,
    maximum_minutes: float | None = None,
) -> dict[int, float]:
    distances = {source: 0.0}
    queue: list[tuple[float, int]] = [(0.0, source)]

    while queue:
        current_distance, node = heapq.heappop(queue)
        if current_distance != distances.get(node):
            continue
        if maximum_minutes is not None and current_distance > maximum_minutes:
            break

        for neighbor, edge_minutes in graph.get(node, []):
            candidate = current_distance + edge_minutes
            if maximum_minutes is not None and candidate > maximum_minutes:
                continue
            if candidate < distances.get(neighbor, float("inf")):
                distances[neighbor] = candidate
                heapq.heappush(queue, (candidate, neighbor))

    return distances


def calculate_coverage(
    con: duckdb.DuckDBPyConnection,
    graph: dict[int, list[tuple[int, float]]],
    station_sources: list[tuple[int, int, float, float]],
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    con.execute("""
        CREATE OR REPLACE TABLE cell_endpoints AS
        SELECT cell_id, start_id AS node_id FROM cell_segments
        UNION
        SELECT cell_id, end_id FROM cell_segments
    """)
    # One row per (cell, station): the fastest time from that station to either
    # endpoint of the cell's snapped segment.
    con.execute(
        "CREATE OR REPLACE TABLE cell_station_minutes (cell_id BIGINT, station_id BIGINT, minutes DOUBLE)"
    )

    sources_by_station: dict[int, list[tuple[int, float]]] = defaultdict(list)
    for node_id, station_id, access_minutes, _distance_km in station_sources:
        sources_by_station[station_id].append((node_id, access_minutes))

    for station_id, sources in sorted(sources_by_station.items()):
        node_ids: list[int] = []
        minutes: list[float] = []
        for source_node, access_minutes in sources:
            for node_id, network_minutes in dijkstra(graph, source_node).items():
                node_ids.append(node_id)
                minutes.append(access_minutes + network_minutes)
        station_node_minutes = pa.table({"node_id": node_ids, "minutes": minutes})
        con.execute("""
            INSERT INTO cell_station_minutes
            SELECT e.cell_id, ?, min(t.minutes)
            FROM cell_endpoints AS e
            JOIN station_node_minutes AS t USING (node_id)
            GROUP BY e.cell_id
        """, [station_id])

    # Every generated cell stays visible. No reachable station (including no
    # snapped road within MAX_SAMPLE_DISTANCE_METRES) → dedicated unreachable
    # class; red is only for reachable >15 min.
    green, yellow, orange = (item["max"] for item in RESPONSE_CLASSIFICATIONS[:3])
    con.execute(f"""
        CREATE OR REPLACE TABLE coverage AS
        WITH per_cell AS (
            SELECT cell_id,
                   min(minutes) AS min_minutes,
                   avg(minutes) AS mean_minutes,
                   count(*) AS reachable_stations
            FROM cell_station_minutes
            GROUP BY cell_id
        )
        SELECT
            c.h3_cell,
            {H3_RESOLUTION} AS h3_resolution,
            CASE
                WHEN p.cell_id IS NULL THEN 'unreachable'
                WHEN p.min_minutes < {green} THEN 'green'
                WHEN p.min_minutes < {yellow} THEN 'yellow'
                WHEN p.min_minutes < {orange} THEN 'orange'
                ELSE 'red'
            END AS response_class,
            round(coalesce(p.min_minutes, {MAX_RESPONSE_MINUTES + 0.01}), 2) AS min_response_minutes,
            round(coalesce(p.mean_minutes, {MAX_RESPONSE_MINUTES + 0.01}), 2) AS mean_response_minutes,
            coalesce(p.reachable_stations, 0) AS reachable_stations,
            CASE WHEN s.cell_id IS NULL THEN 0 ELSE 1 END AS road_segment_count,
            'Road_Network' AS source_layer
        FROM h3_cells AS c
        LEFT JOIN per_cell AS p USING (cell_id)
        LEFT JOIN cell_segments AS s USING (cell_id)
    """)
    result = con.execute("SELECT * FROM coverage ORDER BY h3_cell")
    columns = [column[0] for column in result.description]
    cell_results = [dict(zip(columns, row)) for row in result.fetchall()]

    metadata = {
        "h3_resolution": H3_RESOLUTION,
        "maximum_sample_distance_metres": MAX_SAMPLE_DISTANCE_METRES,
        "maximum_response_minutes": MAX_RESPONSE_MINUTES,
        "speed_profile_kmh": ROAD_SPEED_KMH,
        "classification_thresholds_minutes": {
            "green": "< 5",
            "yellow": "5-10",
            "orange": "10-15",
            "red": "> 15 (reachable only)",
            "unreachable": "no reachable station / no station-reachable road within snap cap",
        },
        "method": (
            "Dense h3.polygon_to_cells polyfill of the WA LGA boundary union at resolution 7; "
            "cell→road snap to nearest station-reachable segment within maximum_sample_distance_metres; "
            "shortest road-network travel time from each police station; "
            "response class uses the fastest reachable station; "
            "unreachable when reachable_stations=0 (not painted red)."
        ),
        "station_to_road_method": "Nearest road segment, connected to both endpoints using the segment speed assumption.",
        "cell_to_road_method": (
            f"Nearest projected road segment on a station-reachable graph component "
            f"within {MAX_SAMPLE_DISTANCE_METRES:.0f} m of the cell centroid; "
            "orphaned components are ignored. Farther cells get an empty assignment "
            "and class unreachable. Exact distance ties go to the lowest segment_id. "
            "No last-mile off-network travel time is added."
        ),
        "h3_fill_method": "h3.polygon_to_cells (LatLngPoly) on the boundary union -- not 0.1 deg lattice + grid_disk.",
        "limitations": [
            "Road speed values are estimates because source data has no speed field.",
            "Station-to-road connection uses the nearest segment and an endpoint-based estimate.",
            "Cell→road snap prefers station-reachable segments within the cap; orphaned components and cells beyond the cap are unreachable.",
            "No last-mile centroid→road travel time is included for snapped cells.",
            "Cells with no reachable station use response_class=unreachable (map: dark grey); red is reachable >15 min only.",
        ],
    }
    return cell_results, metadata

In [ ]:
# The graph and H3 cell assignment are intentionally run separately so that
# failures are easy to diagnose before the potentially larger response run.
con = connect()
load_inputs(con)
graph = build_road_graph(con)
station_sources = create_station_sources(con)
station_reachable_nodes = nodes_reachable_from_stations(graph, station_sources)
full_grid = generate_full_grid(con)
assign_h3_cells(con, station_reachable_nodes)

assigned_count = con.execute("SELECT count(*) FROM cell_segments").fetchone()[0]
unassigned_count = len(full_grid) - assigned_count
print(f"Full-grid H3 cells (dense polyfill): {len(full_grid)}")
print(f"Station-reachable graph nodes: {len(station_reachable_nodes)}")
print(
    f"Cells with station-reachable road snap (≤{MAX_SAMPLE_DISTANCE_METRES:.0f} m): "
    f"{assigned_count}"
)
print(
    f"Cells with no station-reachable snap (unreachable via snap cap): "
    f"{unassigned_count}"
)
print(f"Station-to-road sources: {len(station_sources)}")

In [ ]:
cell_results, metadata = calculate_coverage(con, graph, station_sources)

print(f"Covered H3 cells: {len(cell_results)}")
con.sql("""
    SELECT response_class, count(*) AS cells
    FROM coverage
    GROUP BY response_class
    ORDER BY list_position(['green', 'yellow', 'orange', 'red', 'unreachable'], response_class)
""").show()

In [ ]:
def cell_ring_coordinates(cell: str) -> list[list[float]]:
    """Closed GeoJSON ring [lng, lat] for an H3 cell."""
    ring = [
        [longitude, latitude]
        for latitude, longitude in h3.cell_to_boundary(cell)
    ]
    if ring and ring[0] != ring[-1]:
        ring.append(ring[0])
    return ring


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_GEOJSON = PROCESSED_DIR / "Police_Response_Coverage.geojson"
OUTPUT_PMTILES = PROCESSED_DIR / "Police_Response_Coverage.pmtiles"
OUTPUT_METADATA = PROCESSED_DIR / "Police_Response_Coverage_metadata.json"

coverage_layer = {
    "type": "FeatureCollection",
    "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
    "features": [
        {
            "type": "Feature",
            "properties": {
                **result,
                "source_layer": "Road_Network",
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [cell_ring_coordinates(result["h3_cell"])],
            },
        }
        for result in cell_results
    ],
}

OUTPUT_GEOJSON.write_text(json.dumps(coverage_layer, indent=2), encoding="utf-8")
OUTPUT_METADATA.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Wrote GeoJSON: {OUTPUT_GEOJSON}")
print(f"Wrote metadata: {OUTPUT_METADATA}")

if shutil.which("tippecanoe") is None:
    raise RuntimeError("tippecanoe is not installed. Install it with your system package manager or ensure it is on PATH.")

# M4-validated flags: keep every hex through city zoom (no tiny-polygon attrition).
command = [
    "tippecanoe",
    "-o", str(OUTPUT_PMTILES),
    "-Z", "0",
    "-z", "12",
    "--force",
    "--no-tile-size-limit",
    "--no-feature-limit",
    "--no-tiny-polygon-reduction",
    "--no-tiny-polygon-reduction-at-maximum-zoom",
    "--drop-rate=0",
    "--detect-shared-borders",
    "-l", "Police_Response_Coverage",
    "-y", "h3_cell",
    "-y", "h3_resolution",
    "-y", "response_class",
    "-y", "min_response_minutes",
    "-y", "mean_response_minutes",
    "-y", "reachable_stations",
    "-y", "road_segment_count",
    "-y", "source_layer",
    str(OUTPUT_GEOJSON),
]
result = subprocess.run(command, capture_output=True, text=True, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Tippecanoe failed:\n{result.stdout}\n{result.stderr}")

print(f"Wrote PMTiles: {OUTPUT_PMTILES}")
print(f"PMTiles size: {OUTPUT_PMTILES.stat().st_size} bytes")


## Validation checklist

Before uploading the layer, verify:

- The GeoJSON is a dense H3 res-7 polyfill of the WA boundary (no lattice holes in Perth CBD).
- Polygon rings are closed; every cell has `h3_resolution`, a response class, and a finite response-time field (`unreachable` when `reachable_stations=0` / no station-reachable road within 5 km; red only for reachable >15 min).
- Tippecanoe used `-Z0 -z12` with `--no-tiny-polygon-reduction` and `--drop-rate=0`.
- The metadata records the speed assumptions, dense-fill method, and limitations.
- The PMTiles file can be opened with the map client.
- The output is reviewed with an accurate speed profile before operational use.

The generated files are written to `data/processed` and are excluded from source control by the existing data ignore rules. S3 upload remains a separate, explicit step -- not part of this notebook run.


## Explore the working database

Every intermediate step is a table in `data/processed/Police_Response_Coverage.duckdb` (`roads`, `segments`, `station_sources`, `h3_cells`, `cell_segments`, `cell_station_minutes`, `coverage`, …). Close the notebook's connection (`con.close()`), then open it with the DuckDB CLI or Python, for example:

```sql
LOAD spatial;
SELECT response_class, count(*) FROM coverage GROUP BY ALL;
```